<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-06-advanced-rag-and-open-source-llms/notebooks/Ophelie_Gemini_vs_OpenSource_LLM_Comparison_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task: Evaluate Gemini vs Mistral 7B / Phi-2 / TinyLlama on SDF Doc Queries

**Objective:** hold the RAG retrieval layer completely fixed and swap *only the generator LLM*, so that
every difference we measure is attributable to the model rather than to retrieval.

**Models compared**
| Model | How it runs | Size |
| --- | --- | --- |
| Gemini (`models/gemini-flash-latest`) | Hosted API, needs `GOOGLE_API_KEY` | n/a |
| Mistral-7B-Instruct-v0.2 (Q4_K_M GGUF) | Local, llama.cpp on the Colab GPU | ~4.1 GB |
| Phi-2 (Q4_K_M GGUF) | Local, llama.cpp on the Colab GPU | ~1.8 GB |
| TinyLlama-1.1B-Chat-v1.0 (Q4_K_M GGUF) | Local, llama.cpp on the Colab GPU | ~0.7 GB |

**Experimental design.** One document, one chunking config (SentenceSplitter 200/30), one embedding
model (MiniLM-L6-v2), one retriever (vector, top_k=4). The retrieved chunks are assembled into a single
prompt string, and *that exact same string* is sent to all four models. So retrieval quality is a
constant, not a variable.

**Before running:** Runtime -> Change runtime type -> T4 GPU. Also make sure `GOOGLE_API_KEY` is set in
the Colab Secrets panel (key icon in the left sidebar) with notebook access enabled.

Expect roughly 20-30 minutes end to end, most of it model downloads.

## Step 1: Install packages

In [ ]:
!pip install -q PyMuPDF
!pip install -q llama-index llama-index-embeddings-huggingface llama-index-llms-google-genai
!pip install -q gdown

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU. Set Runtime -> Change runtime type -> T4 GPU, then rerun.")

CUDA available: True
GPU: Tesla T4


In [ ]:
# llama-cpp-python with CUDA 12.x prebuilt wheels (same build used in the previous task)
!pip install -q --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

## Step 2: Load the SDF document

Same source document as every earlier task in this project: the Cytiva AKTA ready flow-kit storage
letter plus two Certificates of Quality (3 pages).

In [ ]:
import os, fitz

PDF_PATH = "/content/sample-sdf-document.pdf"
DRIVE_FILE_ID = "1k_ZRrdSSJLxhMvBnuoaPVYTz05lLmYEN"  # sample-sdf-document (Project 5).pdf

if not os.path.exists(PDF_PATH):
    !gdown "https://drive.google.com/uc?id={DRIVE_FILE_ID}" -O {PDF_PATH} || true

if not os.path.exists(PDF_PATH) or os.path.getsize(PDF_PATH) < 10000:
    print("Automatic download did not work. Run the next cell to upload the PDF by hand.")
else:
    doc = fitz.open(PDF_PATH)
    full_text = "\n".join(page.get_text() for page in doc)
    print(f"Loaded {len(doc)} pages, {len(full_text)} characters")
    print(full_text[:400])

Loaded 3 pages, 4025 characters
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as w


In [ ]:
# FALLBACK ONLY - run this if the gdown cell above failed, then re-run the cell above.
# from google.colab import files
# uploaded = files.upload()   # pick sample-sdf-document.pdf
# import shutil
# shutil.move(list(uploaded.keys())[0], PDF_PATH)

## Step 3: Build the retrieval layer (built once, shared by all four models)

In [ ]:
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import time

Settings.llm = None  # we call each LLM by hand, so LlamaIndex must not try to load a default one
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

splitter = SentenceSplitter(chunk_size=200, chunk_overlap=30)
nodes = splitter.get_nodes_from_documents([Document(text=full_text)])
print(f"{len(nodes)} chunks")

t0 = time.time()
index = VectorStoreIndex(nodes)
print(f"Index built in {time.time() - t0:.3f}s")

retriever = index.as_retriever(similarity_top_k=4)

LLM is explicitly disabled. Using MockLLM.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

6 chunks
Index built in 0.027s


In [ ]:
QUERIES = [
    "What sterilization method was used for this product?",
    "Summarize the key specifications in this document.",
    "What are the storage conditions for this product?",
]

PROMPT_TEMPLATE = (
    "You are answering questions about a pharmaceutical supplier document.\n\n"
    "CONTEXT:\n{context}\n\n"
    "QUESTION: {question}\n\n"
    "Answer the question using only the context above. If the context does not contain the "
    "answer, say so explicitly. Be specific and include exact values where they appear.\n\n"
    "ANSWER:"
)

# Retrieve ONCE per query. Every model gets a byte-identical prompt.
PROMPTS = {}
RETRIEVAL_LOG = {}

for q in QUERIES:
    t0 = time.time()
    hits = retriever.retrieve(q)
    dt = time.time() - t0
    context = "\n\n".join(f"[chunk {i+1}] {h.node.get_content()}" for i, h in enumerate(hits))
    PROMPTS[q] = PROMPT_TEMPLATE.format(context=context, question=q)
    RETRIEVAL_LOG[q] = {
        "retrieve_seconds": round(dt, 4),
        "n_chunks": len(hits),
        "scores": [round(h.score, 4) for h in hits],
        "top_chunk_excerpt": hits[0].node.get_content()[:220],
        "prompt_chars": len(PROMPTS[q]),
    }
    print(f"\nQ: {q}")
    print(f"   {len(hits)} chunks, scores {RETRIEVAL_LOG[q]['scores']}, "
          f"retrieve {dt:.4f}s, prompt {len(PROMPTS[q])} chars")
    print(f"   top chunk: {RETRIEVAL_LOG[q]['top_chunk_excerpt'][:150]}...")


Q: What sterilization method was used for this product?
   4 chunks, scores [0.3662, 0.3626, 0.2977, 0.1885], retrieve 0.0120s, prompt 3318 chars
   top chunk: Description 
Limits 
Autoclave
Pump tubing
121°C > 15 min 
Gamma Irradiation 
Inlets 
25.0 – 40.0 kGy 
Regulatory Conformance 
Bio-safety
All polymeri...

Q: Summarize the key specifications in this document.
   4 chunks, scores [0.2377, 0.1654, 0.1569, 0.1451], retrieve 0.0079s, prompt 3104 chars
   top chunk: Description
Part Number
Operating Temperature
High Flow Kit F, Modified, ÄKTA ready
29477427
+2 C to +40 C
High Flow Gradient C, Modified, ÄKTA ready
...

Q: What are the storage conditions for this product?
   4 chunks, scores [0.241, 0.1617, 0.1581, 0.1173], retrieve 0.0067s, prompt 3380 chars
   top chunk: Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It ...


## Step 4: Scoring helpers

`latency`, `tokens/sec` and `finish_reason` are measured directly. Fact recall is a crude keyword
proxy - it checks whether the values that genuinely appear in the source document show up in the
answer. It is a signal, not a grade: read the actual answers before trusting it.

In [ ]:
EXPECTED_FACTS = {
    QUERIES[0]: ["gamma", "irradiat", "kgy", "autoclave", "121"],
    QUERIES[1]: ["flow", "lot", "temperature", "irradiat", "usp"],
    QUERIES[2]: ["+5", "storage", "+2", "+40", "brittle"],
}

RESULTS = []  # one row per (model, query)

def fact_recall(query, answer):
    keys = EXPECTED_FACTS[query]
    low = answer.lower()
    hit = [k for k in keys if k.lower() in low]
    return len(hit), len(keys), hit

def record(model, query, answer, seconds, n_tokens, finish_reason, notes=""):
    hits, total, which = fact_recall(query, answer)
    row = {
        "model": model,
        "query": query,
        "answer": answer.strip(),
        "seconds": round(seconds, 2),
        "output_tokens": n_tokens,
        "tokens_per_sec": round(n_tokens / seconds, 2) if (n_tokens and seconds) else None,
        "answer_chars": len(answer.strip()),
        "finish_reason": finish_reason,
        "fact_recall": f"{hits}/{total}",
        "facts_found": which,
        "notes": notes,
    }
    RESULTS.append(row)
    print(f"\n=== {model} | {query}")
    print(f"    {row['seconds']}s | {n_tokens} tok | {row['tokens_per_sec']} tok/s | "
          f"finish={finish_reason} | facts {row['fact_recall']} {which}")
    print(f"    {row['answer'][:600]}")
    return row

MAX_TOKENS = 300
TEMPERATURE = 0.1

## Step 5: Gemini (hosted API baseline)

In [ ]:
from google.colab import userdata
from google.colab import userdata
RESULTS = [r for r in RESULTS if not str(r.get('model')).startswith('Gemini')]
exec("from google import genai\nfrom google.genai import types\nclient = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))\nGM = 'models/gemini-flash-lite-latest'\nfor q in QUERIES:\n    t0 = time.time()\n    resp = client.models.generate_content(model=GM, contents=PROMPTS[q], config=types.GenerateContentConfig(temperature=TEMPERATURE, max_output_tokens=MAX_TOKENS))\n    dt = time.time() - t0\n    cand = resp.candidates[0]\n    parts = cand.content.parts or []\n    text = ''.join([(p.text or '') for p in parts])\n    n_tok = resp.usage_metadata.candidates_token_count\n    finish = str(cand.finish_reason)\n    record('Gemini Flash Lite', q, text, dt, n_tok, finish, notes='hosted API ' + GM + ' via google-genai SDK, real token counts')\n")from llama_index.llms.google_genai import GoogleGenAI
import nest_asyncio
nest_asyncio.apply()
RESULTS = [r for r in RESULTS if not str(r.get('model')).startswith('Gemini')]

gemini = GoogleGenAI(
    model="models/gemini-flash-lite-latest",
    api_key=userdata.get("GOOGLE_API_KEY"),
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

for q in QUERIES:
    t0 = time.time()
    resp = gemini.complete(PROMPTS[q])
    dt = time.time() - t0
    text = str(resp)
    # Gemini token accounting via the raw response when available
    n_tok, finish = None, "unknown"
    try:
        raw = resp.raw
        n_tok = raw.usage_metadata.candidates_token_count
        finish = str(raw.candidates[0].finish_reason)
    except Exception:
        n_tok = int(len(text.split()) * 1.3)  # rough fallback
        finish = "unknown (estimated tokens)"
    record("Gemini Flash Lite", q, text, dt, n_tok, finish, notes="hosted API models/gemini-flash-lite-latest, network round-trip included, output tokens estimated")

SyntaxError: invalid syntax (4004139745.py, line 4)

## Step 6: The three local GGUF models

Each model is downloaded, loaded with **full GPU offload** (`n_gpu_layers=-1`), run against all three
prompts, then freed before the next one loads. Full offload is a deliberate change from the previous
task, which used `n_gpu_layers=1` and left Mistral running mostly on CPU at ~1.6 tokens/sec.

In [ ]:
import gc
from llama_cpp import Llama

MODELS = [
    {
        "name": "Mistral-7B-Instruct-v0.2 Q4_K_M",
        "path": "/content/mistral.gguf",
        "url": "https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf",
        "mode": "chat",
        "n_ctx": 4096,   # Mistral v0.2 trained on 32k, 4096 is plenty here
    },
    {
        "name": "Phi-2 Q4_K_M",
        "path": "/content/phi2.gguf",
        "url": "https://huggingface.co/TheBloke/phi-2-GGUF/resolve/main/phi-2.Q4_K_M.gguf",
        "mode": "completion",   # Phi-2 is a base model with no chat template
        "n_ctx": 2048,          # Phi-2's native training context
    },
    {
        "name": "TinyLlama-1.1B-Chat-v1.0 Q4_K_M",
        "path": "/content/tinyllama.gguf",
        "url": "https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
        "mode": "chat",
        "n_ctx": 2048,          # TinyLlama's native training context
    },
]

for m in MODELS:
    if not os.path.exists(m["path"]):
        print(f"Downloading {m['name']} ...")
        !wget -q --show-progress {m['url']} -O {m['path']}
    print(f"{m['name']}: {os.path.getsize(m['path']) / (1024**3):.2f} GB")

In [ ]:
def run_local_model(spec):
    print(f"\n{'#' * 70}\n# Loading {spec['name']}\n{'#' * 70}")
    t_load = time.time()
    llm = Llama(
        model_path=spec["path"],
        n_gpu_layers=-1,          # full offload to the T4
        n_ctx=spec["n_ctx"],
        verbose=False,
    )
    load_seconds = time.time() - t_load
    print(f"loaded in {load_seconds:.2f}s, context window {spec['n_ctx']} tokens")

    for q in QUERIES:
        prompt = PROMPTS[q]
        t0 = time.time()
        if spec["mode"] == "chat":
            out = llm.create_chat_completion(
                messages=[{"role": "user", "content": prompt}],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
            )
            text = out["choices"][0]["message"]["content"]
        else:
            # Phi-2's documented Instruct/Output format
            out = llm(
                f"Instruct: {prompt}\nOutput:",
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                stop=["Instruct:", "\n\n\n"],
            )
            text = out["choices"][0]["text"]
        dt = time.time() - t0
        n_tok = out["usage"]["completion_tokens"]
        finish = out["choices"][0].get("finish_reason", "unknown")
        record(spec["name"], q, text, dt, n_tok, finish,
               notes=f"local GGUF, full GPU offload, n_ctx={spec['n_ctx']}, load {load_seconds:.1f}s")

    del llm
    gc.collect()
    torch.cuda.empty_cache()
    print(f"released {spec['name']}")

for spec in MODELS:
    run_local_model(spec)

## Step 7: Context-limit test

The three questions above sit comfortably inside every model's window. This cell does the opposite:
it stuffs the **entire document** into the prompt and asks for a long structured summary, so we can
see which models truncate (`finish_reason == "length"`) and which handle it.

In [ ]:
LONG_PROMPT = (
    "You are reading a full pharmaceutical supplier document.\n\n"
    "DOCUMENT:\n" + full_text + "\n\n"
    "Produce a detailed structured summary covering: every product named, every lot and article "
    "number, every date, every test method with its limit and result, the regulatory conformance "
    "statements, and the storage and operating temperatures.\n\nSUMMARY:"
)
print(f"Long prompt: {len(LONG_PROMPT)} chars")

CONTEXT_RESULTS = []
LONG_MAX_TOKENS = 600

gemini_long = GoogleGenAI(
    model="models/gemini-flash-latest",
    api_key=userdata.get("GOOGLE_API_KEY"),
    temperature=TEMPERATURE,
    max_tokens=LONG_MAX_TOKENS,
)

# Gemini
t0 = time.time()
resp = gemini_long.complete(LONG_PROMPT)
dt = time.time() - t0
try:
    finish = str(resp.raw.candidates[0].finish_reason)
    n_tok = resp.raw.usage_metadata.candidates_token_count
except Exception:
    finish, n_tok = "unknown", None
CONTEXT_RESULTS.append({"model": "Gemini Flash", "seconds": round(dt, 2),
                        "output_tokens": n_tok, "finish_reason": finish,
                        "answer_chars": len(str(resp)), "answer": str(resp)})
print(f"Gemini: {dt:.2f}s, finish={finish}, {len(str(resp))} chars")

# Local models, reloaded one at a time
for spec in MODELS:
    llm = Llama(model_path=spec["path"], n_gpu_layers=-1, n_ctx=spec["n_ctx"], verbose=False)
    t0 = time.time()
    try:
        if spec["mode"] == "chat":
            out = llm.create_chat_completion(
                messages=[{"role": "user", "content": LONG_PROMPT}],
                max_tokens=LONG_MAX_TOKENS, temperature=TEMPERATURE)
            text = out["choices"][0]["message"]["content"]
        else:
            out = llm(f"Instruct: {LONG_PROMPT}\nOutput:", max_tokens=LONG_MAX_TOKENS,
                      temperature=TEMPERATURE, stop=["Instruct:"])
            text = out["choices"][0]["text"]
        dt = time.time() - t0
        CONTEXT_RESULTS.append({
            "model": spec["name"], "seconds": round(dt, 2),
            "context_window": spec["n_ctx"],
            "output_tokens": out["usage"]["completion_tokens"],
            "prompt_tokens": out["usage"]["prompt_tokens"],
            "finish_reason": out["choices"][0].get("finish_reason"),
            "answer_chars": len(text), "answer": text})
        print(f"{spec['name']} (window {spec['n_ctx']}): {dt:.2f}s, "
              f"finish={out['choices'][0].get('finish_reason')}, "
              f"{out['usage']['prompt_tokens']} prompt tok, {len(text)} chars")
    except Exception as e:
        # An overflow here is itself a result: the model's window could not hold the document.
        CONTEXT_RESULTS.append({"model": spec["name"], "context_window": spec["n_ctx"],
                                "error": f"{type(e).__name__}: {e}"})
        print(f"{spec['name']} (window {spec['n_ctx']}) FAILED: {type(e).__name__}: {e}")
    del llm
    gc.collect()
    torch.cuda.empty_cache()

## Step 8: Results summary

In [ ]:
import pandas as pd

df = pd.DataFrame(RESULTS)
summary = (df.groupby("model")
             .agg(avg_seconds=("seconds", "mean"),
                  avg_tokens_per_sec=("tokens_per_sec", "mean"),
                  avg_answer_chars=("answer_chars", "mean"),
                  truncated=("finish_reason", lambda s: sum("length" in str(x).lower() for x in s)))
             .round(2))
recall = df.groupby("model")["fact_recall"].apply(list)
summary["fact_recall_per_query"] = recall
display(summary)
display(df[["model", "query", "seconds", "tokens_per_sec", "fact_recall", "finish_reason"]])

In [ ]:
# Full dump - copy everything printed below into the chat to have the write-up built from it.
import json

full_payload = {
    "retrieval": RETRIEVAL_LOG,
    "per_query_results": RESULTS,
    "context_limit_test": CONTEXT_RESULTS,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only",
}
with open("/content/model_comparison_results.json", "w") as f:
    json.dump(full_payload, f, indent=2)

def trim(d, key, n=700):
    d = dict(d)
    if key in d and isinstance(d[key], str) and len(d[key]) > n:
        d[key] = d[key][:n] + f" ...[TRUNCATED, {len(d[key])} chars total]"
    return d

compact = {
    "gpu": full_payload["gpu"],
    "retrieval": RETRIEVAL_LOG,
    "per_query_results": [trim(r, "answer") for r in RESULTS],
    "context_limit_test": [trim(c, "answer", 500) for c in CONTEXT_RESULTS],
}

print("=" * 78)
print("COPY EVERYTHING BELOW THIS LINE")
print("=" * 78)
print(json.dumps(compact, indent=1))
print("=" * 78)
print("END OF RESULTS")

## Observations

_Fill in after running - or paste the JSON dump above into the chat and have it drafted from the real
numbers._

- Speed:
- Accuracy / fact recall:
- Context limit:
- Which model felt most "human":
- Would you trust it with important documents: